# Sorting and Limiting — SQLite Demo

Run the cells to explore ORDER BY, tie-breakers, NULL placement, LIMIT/OFFSET, and keyset pagination.

In [1]:
import sqlite3
import pandas as pd
con = sqlite3.connect(':memory:')
print(sqlite3.sqlite_version)

3.46.1


## 1. Create and populate sample data

In [2]:
con.executescript("""
CREATE TABLE products (
  product_id INTEGER PRIMARY KEY,
  product_name TEXT NOT NULL,
  category TEXT,
  price REAL
);
CREATE TABLE students (
  student_id INTEGER PRIMARY KEY,
  student_name TEXT NOT NULL,
  department TEXT,
  age INTEGER,
  score REAL
);
CREATE TABLE orders (
  order_id INTEGER PRIMARY KEY,
  customer_id INTEGER,
  order_date TEXT,
  total REAL
);
INSERT INTO products VALUES
(1, 'Mouse', 'Accessories', 20),
(2, 'Laptop', 'Computers', 900),
(3, 'Keyboard', 'Accessories', 40),
(4, 'Tablet', 'Computers', 300),
(5, 'Cable', 'Accessories', 10),
(6, 'Monitor', 'Computers', 300),
(7, 'Mystery Item', 'Other', NULL);
INSERT INTO students VALUES
(1, 'Asha', 'Data Science', 20, 91),
(2, 'Ravi', 'Computer Science', 22, 84),
(3, 'Anaya', 'Data Science', 19, 91),
(4, 'Kabir', 'Cybersecurity', 25, NULL),
(5, 'Mina', 'Computer Science', 22, 77);
INSERT INTO orders VALUES
(101, 1, '2026-01-05', 250),
(102, 2, '2026-02-14', 1200),
(103, 1, '2026-02-20', 90),
(104, 3, '2025-12-31', 75),
(105, 2, '2026-03-01', 400);
""")
pd.read_sql_query("SELECT * FROM products", con)

,product_id,product_name,category,price
0,1,Mouse,Accessories,20.0
1,2,Laptop,Computers,900.0
2,3,Keyboard,Accessories,40.0
3,4,Tablet,Computers,300.0
4,5,Cable,Accessories,10.0
5,6,Monitor,Computers,300.0
6,7,Mystery Item,Other,NaN


## 2. Ascending, descending, and multi-column ordering

In [3]:
pd.read_sql_query("""
SELECT product_name, category, price
FROM products
ORDER BY category ASC, price DESC, product_id ASC
""", con)

,product_name,category,price
0,Keyboard,Accessories,40.0
1,Mouse,Accessories,20.0
2,Cable,Accessories,10.0
3,Laptop,Computers,900.0
4,Tablet,Computers,300.0
5,Monitor,Computers,300.0
6,Mystery Item,Other,NaN


## 3. Top-N with deterministic tie-breakers

In [4]:
pd.read_sql_query("""
SELECT product_id, product_name, price
FROM products
ORDER BY price DESC, product_id ASC
LIMIT 3
""", con)

,product_id,product_name,price
0,2,Laptop,900.0
1,4,Tablet,300.0
2,6,Monitor,300.0


## 4. NULL values last

In [5]:
pd.read_sql_query("""
SELECT product_name, price
FROM products
ORDER BY CASE WHEN price IS NULL THEN 1 ELSE 0 END,
         price DESC,
         product_id ASC
""", con)

,product_name,price
0,Laptop,900.0
1,Tablet,300.0
2,Monitor,300.0
3,Keyboard,40.0
4,Mouse,20.0
5,Cable,10.0
6,Mystery Item,NaN


## 5. Pagination with LIMIT and OFFSET

In [6]:
pd.read_sql_query("""
SELECT product_id, product_name
FROM products
ORDER BY product_id
LIMIT 3 OFFSET 2
""", con)

,product_id,product_name
0,3,Keyboard
1,4,Tablet
2,5,Cable


## 6. Keyset pagination

In [7]:
last_seen_id = 3
pd.read_sql_query("""
SELECT product_id, product_name
FROM products
WHERE product_id > ?
ORDER BY product_id
LIMIT 3
""", con, params=(last_seen_id,))

,product_id,product_name
0,4,Tablet
1,5,Cable
2,6,Monitor


## 7. Most recent order per customer

In [8]:
pd.read_sql_query("""
WITH ranked_orders AS (
  SELECT *,
         ROW_NUMBER() OVER (
           PARTITION BY customer_id
           ORDER BY order_date DESC, order_id DESC
         ) AS row_num
  FROM orders
)
SELECT order_id, customer_id, order_date, total
FROM ranked_orders
WHERE row_num = 1
ORDER BY customer_id
""", con)

,order_id,customer_id,order_date,total
0,103,1,2026-02-20,90.0
1,105,2,2026-03-01,400.0
2,104,3,2025-12-31,75.0


## 8. Sorting scores and preserving ties

In [9]:
pd.read_sql_query("""
SELECT student_name, score
FROM students
ORDER BY CASE WHEN score IS NULL THEN 1 ELSE 0 END,
         score DESC,
         student_name ASC
""", con)

,student_name,score
0,Anaya,91.0
1,Asha,91.0
2,Ravi,84.0
3,Mina,77.0
4,Kabir,NaN


## 9. Random sample (SQLite)

In [10]:
pd.read_sql_query("SELECT product_name FROM products ORDER BY RANDOM() LIMIT 3", con)

,product_name
0,Tablet
1,Monitor
2,Cable


## 10. Close the database

In [11]:
con.close()
print("SQLite connection closed.")

SQLite connection closed.
